In [1]:
from collections import Counter
from pathlib import Path
import pickle, re, pandas as pd
import automated_llm_probes as alp

TARGET_N = 700
LOCK20 = [
    "claude-haiku-4.5", "claude-opus-4.5", "claude-opus-4.7", "claude-opus-5",
    "claude-sonnet-4.5", "gpt-3.5-turbo", "gpt-4-turbo", "gpt-4o", "gpt-4o-mini",
    "gpt-5.4", "gpt-5.6-sol", "grok-4.2", "grok-4.3", "grok-4.5", "grok-4.6",
    "grok-build-0.1", "llama-3.1-8b", "llama-3.2-3b", "llama-4-maverick", "llama-4-scout"]
HUMAN_N = {
    "brick": 2019, "knife": 1028, "car tires": 960, "box": 833, "rope": 829,
    "pen": 742, "wooden slat": 671, "paperclip": 534, "tin can": 425,
    "socks": 339, "light bulb": 337, "spoon": 337, "towel": 327, "book": 326,
    "belt": 300, "bucket": 300, "sock": 300, "candle": 299}

def targets(n, human_n):
    tot = sum(human_n.values())
    raw = {c: n * k / tot for c, k in human_n.items()}
    out = {c: int(v) for c, v in raw.items()}
    for c in sorted(raw, key=lambda c: raw[c] - out[c], reverse=True):
        if sum(out.values()) >= n:
            break
        out[c] += 1
    return out

def slug(name):
    return re.sub(r"[^\w\-.]+", "-", str(name).strip()).strip("-").lower()

def model_dir(task, name):
    s = slug(name)
    for root in (Path("data") / task / s, Path(task) / s):
        if root.exists():
            return root
    return Path("data") / task / s

def cue_of(row):
    cue = (row.get("kwargs") or {}).get("cue") or row.get("cue") or row.get("object")
    if isinstance(cue, (list, tuple)):
        cue = " ".join(str(x) for x in cue if str(x).strip())
    cue = str(cue).strip().lower() if cue else ""
    if not cue:
        m = re.search(r"object:\s*(.+?)\s*\?", str(row.get("prompt") or ""), re.I)
        if m:
            cue = m.group(1).strip().lower()
    return cue

def load_row(p):
    try:
        row = pickle.load(open(p, "rb"))
    except Exception:
        return None
    if row.get("error") or not row.get("raw"):
        return None
    return row

tgt = targets(TARGET_N, HUMAN_N)
print("AUT targets", tgt, "sum", sum(tgt.values()))

seen = {}
for m in alp.ready_models():
    if m["name"] in LOCK20 and m["name"] not in seen:
        seen[m["name"]] = m
models = [seen[n] for n in LOCK20 if n in seen]
print("ready", [m["name"] for m in models])
print("not ready", [n for n in LOCK20 if n not in seen])

for m in models:
    have = Counter()
    root = model_dir("aut", m["name"])
    for p in root.rglob("*.pickle"):
        row = load_row(p)
        if not row:
            continue
        c = cue_of(row)
        if c:
            have[c] += 1
    print(f"\n{m['name']}  {sum(have.values())} files  {root}")
    for cue, want in tgt.items():
        gap = max(0, want - have.get(cue, 0))
        print(f"  {cue:16s} {have.get(cue, 0):4d}/{want:<3d}  {'ok' if gap == 0 else f'+{gap}'}")
        if gap:
            alp.collect("AUT", models=[m], n_per_model=gap, cue=cue, n_to_topup=True)
            have[cue] += gap

AUT targets {'brick': 130, 'knife': 66, 'car tires': 62, 'box': 53, 'rope': 53, 'pen': 48, 'wooden slat': 43, 'paperclip': 34, 'tin can': 27, 'socks': 22, 'light bulb': 22, 'spoon': 22, 'towel': 21, 'book': 21, 'belt': 19, 'bucket': 19, 'sock': 19, 'candle': 19} sum 700
ready ['claude-haiku-4.5', 'claude-opus-4.5', 'claude-opus-4.7', 'claude-opus-5', 'claude-sonnet-4.5', 'gpt-3.5-turbo', 'gpt-4-turbo', 'gpt-4o', 'gpt-4o-mini', 'gpt-5.4', 'gpt-5.6-sol', 'grok-4.2', 'grok-4.3', 'grok-4.5', 'grok-4.6', 'grok-build-0.1', 'llama-3.1-8b', 'llama-3.2-3b', 'llama-4-maverick', 'llama-4-scout']
not ready []

claude-haiku-4.5  830 files  data/aut/claude-haiku-4.5
  brick             111/130  +19
  claude-haiku-4.5: 830 collected, 19 to collect


AUT: 100%|███████████████████████████████████████████████████████████████| 19/19 [02:22<00:00,  7.51s/it]


  knife              57/66   +9
  claude-haiku-4.5: 849 collected, 9 to collect


AUT: 100%|█████████████████████████████████████████████████████████████████| 9/9 [01:09<00:00,  7.69s/it]


  car tires          53/62   +9
  claude-haiku-4.5: 858 collected, 9 to collect


...

AUT: 100%|█████████████████████████████████████████████████████████████████| 3/3 [00:14<00:00,  4.78s/it]


  bucket             18/19   +1
  llama-4-scout: 906 collected, 1 to collect


AUT: 100%|█████████████████████████████████████████████████████████████████| 1/1 [00:10<00:00, 10.46s/it]


  sock               19/19   ok
  candle             17/19   +2
  llama-4-scout: 907 collected, 2 to collect


AUT: 100%|█████████████████████████████████████████████████████████████████| 2/2 [00:10<00:00,  5.39s/it]


### Check the distribution of errors by models and cues

In [2]:
from collections import Counter
from pathlib import Path
import pickle,tqdm

def cue(row):
    c = (row.get("kwargs") or {}).get("cue") or row.get("cue") or ""
    return " ".join(c) if isinstance(c, (list, tuple)) else str(c).strip().lower()

err = Counter()
for p in tqdm.tqdm(Path("data/aut").rglob("*.pickle")):
    try:
        row = pickle.load(p.open("rb"))
    except Exception:
        err["UNREADABLE", "?"] += 1
        continue
    if row.get("error") or not row.get("raw"):
        err[row.get("model_name") or "?", cue(row) or "?"] += 1
print("errored", sum(err.values()))
for (m, c), n in err.most_common():
    print(f"{n:5d}  {m:24s}  {c}")

18785it [01:10, 264.63it/s]

errored 0


### Check the distribution of good pickles

In [10]:
have = Counter()
for p in tqdm.tqdm(Path("data/aut").rglob("*.pickle")):
    try:
        row = pickle.load(p.open("rb"))
    except Exception:
        continue
    if row.get("error") or not row.get("raw"):
        continue
    have[row.get("model_name") or "?", cue(row) or "?"] += 1
pd.Series(have,dtype="int64").unstack(
    fill_value=0).sort_index().sort_index(axis=1)

18785it [00:13, 1399.28it/s]


,baseball,belt,book,box,brick,broom,bucket,candle,car tires,clock,...,pencil,pillow,purse,rope,sock,socks,spoon,tin can,towel,wooden slat
claude-haiku-4.5,17,19,30,53,130,17,19,19,62,16,...,21,16,18,53,19,22,30,30,30,43
claude-opus-4.5,15,19,30,53,130,17,19,19,62,15,...,18,16,16,53,19,22,30,30,30,43
claude-opus-4.7,34,42,21,53,130,23,58,42,62,41,...,34,30,28,53,65,22,22,27,21,43
claude-opus-5,30,49,21,53,130,30,48,47,62,30,...,27,32,25,53,68,22,22,27,21,43
claude-sonnet-4.5,16,19,30,53,130,15,19,19,62,19,...,15,16,17,53,19,22,30,30,30,43
deepseek-2.5-chat,14,14,0,0,14,14,14,14,0,14,...,14,14,14,0,14,0,0,0,0,0
deepseek-3.2,14,14,0,0,14,14,14,14,0,14,...,14,14,14,0,14,0,0,0,0,0
deepseek-4-flash-0731,1,1,0,0,0,2,1,1,0,0,...,2,2,1,0,2,0,0,0,0,0
deepseek-4-pro,3,3,0,0,0,3,2,2,0,1,...,3,3,2,0,3,0,0,0,0,0
deepseek-r1,3,3,0,0,4,3,4,3,0,3,...,3,3,3,0,4,0,0,0,0,0


### Backfill pickles without `parsed` and `score` fields

In [ ]:
import os, pickle
import automated_intelligence_tests as ait
from IPython.display import clear_output

def list_pickle_fps(root):
    fps = []
    for dp, _, fns in os.walk(root):
        for n in fns:
            if n.endswith(".pickle") and not n.endswith(".pickle.tmp"):
                fps.append(os.path.join(dp, n))
    return fps

def dump(p, row):
    tmp = p + ".tmp"
    with open(tmp, "wb") as f:
        pickle.dump(row, f, protocol=pickle.HIGHEST_PROTOCOL)
    os.replace(tmp, p)

def load(p):
    with open(p, "rb") as f:
        return pickle.load(f)

fps = list_pickle_fps("./data/aut/")
n = len(fps)
for i, p in enumerate(fps, 1):
    row = load(p)
    if isinstance(row.get("score"), (int, float)):
        clear_output(wait=True)
        print(f"{i}/{n}  skip score={row['score']}")
        continue
    raw = row.get("raw")
    try:
        parsed = ait.parse("aut", raw, stim=row.get("kwargs")) if raw else None
        score = ait.evaluate("aut", parsed).get("score") if parsed else None
    except Exception:
        parsed, score = None, None
    row["parsed"] = parsed
    row["score"] = score
    dump(p, row)
    raw_show = " ".join(str(raw or "").split())[:120]
    clear_output(wait=True)
    print(f"{i}/{n}  score={score}  raw={raw_show}")

6381/17506  skip score=1.0037744816211052
